# 14.03.04 RRT* 실습 — 샘플링 기반 최적 경로 계획

**목표:**  
RRT* (Rapidly-exploring Random Tree Star) 알고리즘의 핵심 원리를 Python으로 직접 구현하고,  
랜덤 샘플링 → 최근접 노드 탐색 → 확장 → **최적 부모 선택 (Choose Parent)** → **재배선 (Rewire)** → 트리 성장의 전 과정을 시각적으로 체감합니다.

- 연속 공간(Continuous Space) 기반 **최적** 경로 탐색
- 100×100 Grid (0.1 m/cell → 10 m × 10 m)
- A*와 달리 **Grid 해상도에 얽매이지 않고** 자유롭게 샘플링
- RRT 대비 **Rewiring**을 통해 점진적으로 최적 경로에 수렴
- **랜덤성** 때문에 매 실행마다 다른 트리가 생성되나, RRT보다 짧은 경로 보장

### RRT* 핵심 흐름:
1. **샘플링 ($q_{rand}$):** 전체 공간에서 무작위 좌표 하나를 선택
2. **가장 가까운 노드 탐색 ($q_{near}$):** 현재 트리에 있는 노드 중 $q_{rand}$와 가장 가까운 노드를 탐색
3. **확장 ($q_{new}$):** $q_{near}$에서 $q_{rand}$ 방향으로 일정 거리($\Delta q$)만큼 이동하여 새로운 노드 $q_{new}$를 생성
4. **충돌 체크:** $q_{near}$에서 $q_{new}$로 가는 직선 경로에 장애물이 있는지 확인
5. **최적 부모 선택 (Choose Parent):** $q_{new}$ 주변 반경 내의 노드들 중, $q_{new}$까지의 **누적 비용(cost)이 가장 작은** 노드를 진짜 부모로 선택
6. **재배선 (Rewire):** $q_{new}$를 통해 기존 주변 노드들의 cost가 더 줄어든다면, 그 노드들의 부모를 $q_{new}$로 변경
7. **반복:** 목적지 근처에 도달할 때까지 위 과정을 반복

> **RRT vs RRT*:** RRT는 단순히 가장 가까운 노드에 연결하지만, RRT*는 **주변 노드들을 살펴 최적의 부모를 선택**하고, **기존 노드들의 연결도 개선**하여 점점 더 짧은 경로를 만듭니다.

## Step 1 — 환경 설정 & Occupancy Grid Map 생성

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle, FancyBboxPatch
from matplotlib.colors import LinearSegmentedColormap
import time
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# --- 맵 상수 ---
GRID_W, GRID_H = 100, 100      # 100 x 100 cell
RES = 0.1                       # 0.1 m/cell  → 전체 10m x 10m
MAP_W, MAP_H = GRID_W * RES, GRID_H * RES  # 10.0 x 10.0 m

# --- 복잡한 장애물 정의 (미로 + 방 + 원형 기둥) ---
# 각 값은 미터(m) 단위. (x, y)는 맵 좌측하단 기준.
OBSTACLES_RECT = [
    # 외곽 벽 (두께 1cell = 0.1m)
    (0.0, 0.0, 10.0, 0.2),      # 하단 벽
    (0.0, 9.8, 10.0, 0.2),      # 상단 벽
    (0.0, 0.0, 0.2, 10.0),      # 좌측 벽
    (9.8, 0.0, 0.2, 10.0),      # 우측 벽

    # 중앙 미로 벽들
    (3.0, 0.0, 0.2, 5.5),       # 수직 벽 1
    (6.0, 4.5, 0.2, 5.5),       # 수직 벽 2
    (1.5, 3.0, 3.0, 0.2),       # 수평 벽 1
    (5.5, 6.0, 3.5, 0.2),       # 수평 벽 2
    (0.0, 7.0, 2.5, 0.2),       # 수평 벽 3
    (7.5, 2.0, 2.5, 0.2),       # 수평 벽 4

    # 방 난간 / 추가 장애물
    (4.0, 4.0, 1.5, 1.5),       # 중앙 사각형 장애물
    (1.0, 5.5, 1.0, 1.0),       # 좌측 방 장애물
    (8.0, 7.5, 1.0, 1.0),       # 우측 상단 장애물
]

OBSTACLES_CIRCLE = [
    {'cx': 2.5, 'cy': 1.5, 'r': 0.6},
    {'cx': 7.0, 'cy': 5.0, 'r': 0.7},
    {'cx': 4.5, 'cy': 8.0, 'r': 0.5},
    {'cx': 8.5, 'cy': 3.0, 'r': 0.4},
]


def build_occupancy_grid():
    """0: free, 100: occupied"""
    grid = np.zeros((GRID_H, GRID_W), dtype=np.int8)
    for oy in range(GRID_H):
        for ox in range(GRID_W):
            x = (ox + 0.5) * RES
            y = (oy + 0.5) * RES
            occ = False
            # 사각형 장애물
            for rx, ry, rw, rh in OBSTACLES_RECT:
                if rx <= x <= rx + rw and ry <= y <= ry + rh:
                    occ = True
                    break
            # 원형 장애물
            if not occ:
                for c in OBSTACLES_CIRCLE:
                    if np.hypot(x - c['cx'], y - c['cy']) <= c['r']:
                        occ = True
                        break
            grid[oy, ox] = 100 if occ else 0
    return grid


occupancy_grid = build_occupancy_grid()


def draw_map(ax, title="Occupancy Grid", show_start_goal=True):
    """장애물(검정) / free(흰) 표시"""
    ax.imshow(occupancy_grid, origin='lower', cmap='gray_r',
              extent=[0, MAP_W, 0, MAP_H], vmin=0, vmax=100)
    if show_start_goal:
        ax.scatter(*START_M, c='lime', s=200, marker='o',
                   edgecolors='black', zorder=5, label='Start')
        ax.scatter(*GOAL_M, c='red', s=200, marker='X',
                   edgecolors='black', zorder=5, label='Goal')
        ax.legend(loc='upper right', fontsize=8)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
    ax.set_xlim(0, MAP_W); ax.set_ylim(0, MAP_H)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)


# 시작/목적지 위치 (미터) — 복잡한 맵의 반대편으로 설정
START_M = (1.0, 1.0)   # 좌측 하단
GOAL_M = (9.0, 9.0)    # 우측 상단

fig, ax = plt.subplots(figsize=(6, 6))
draw_map(ax, "Step 1 — Complex Occupancy Grid Map")
plt.tight_layout()
plt.show()

print("✅ Step 1 완료 — 100×100 (0.1m/cell) 복잡 맵 생성")
print(f"   시작점: {START_M} | 목적지: {GOAL_M}")

## Step 2 — RRT* 기반 설정

### RRT* 핵심 파라미터
| 파라미터 | 설명 |
|---------|------|
| `MAX_ITER` | 최대 반복 횟수 (트리 확장 시도 횟수) |
| `STEP_SIZE` | $\Delta q$ — 한 번 확장 시 이동 거리 (m) |
| `GOAL_THRESHOLD` | 목적지 근처로 간주할 거리 (m) |
| `GOAL_SAMPLE_RATE` | 목적지를 $q_{rand}$로 선택할 확률 (%) |
| `REWIRE_RADIUS` | **RRT* 전용** — Choose Parent / Rewire를 수행할 주변 탐색 반경 (m) |

> **Rewire Radius:** 이 반경 내의 노드들만 부모 후보로 고려하거나 재배선 대상으로 삼습니다.  
> 반경이 너무 작으면 최적화 효과가 줄어들고, 너무 크면 계산량이 증가합니다.

In [ ]:
# RRT* 파라미터
MAX_ITER = 2000                # 최대 반복 횟수
STEP_SIZE = 0.5                # Δq: 한 번에 확장하는 거리 (m)
GOAL_THRESHOLD = 0.5           # 목적지 도달 판정 거리 (m)
GOAL_SAMPLE_RATE = 10          # 무작위 샘플링 점(q_rand) 대신 실제 목적지(Goal)를 선택할 확률 (%) (Goal Bias)
REWIRE_RADIUS = 1.5            # RRT* 전용: Choose Parent / Rewire 주변 탐색 반경 (m)


class RRTStarNode:
    """RRT* 트리의 노드 (cost 필드 추가)"""
    def __init__(self, x, y, parent_idx=None):
        self.x = x
        self.y = y
        self.parent_idx = parent_idx   # 부모 노드 인덱스 (None이면 루트)
        self.cost = 0.0                # 시작점부터 해당 노드까지의 누적 비용 (m)

    def __repr__(self):
        return f"RRTStarNode({self.x:.2f}, {self.y:.2f}, cost={self.cost:.2f})"


def m_to_idx(x_m, y_m):
    """
    미터 좌표 → 그리드 인덱스 (클리핑 포함)
    
    미터 단위의 연속 좌표를 지도 해상도(RES)로 나누어 그리드의 정수 인덱스로 변환합니다.
    인덱스가 지도의 크기(GRID_W, GRID_H)를 벗어나지 않도록 np.clip으로 제한합니다.
    """
    ix = int(np.clip(np.floor(x_m / RES), 0, GRID_W - 1))
    iy = int(np.clip(np.floor(y_m / RES), 0, GRID_H - 1))
    return ix, iy


def idx_to_m(ix, iy):
    """
    그리드 인덱스 → 셀 중심 미터 좌표
    
    그리드 맵의 정수 인덱스를 미터 단위 좌표로 변환합니다.
    셀의 경계가 아닌 정중앙을 나타내기 위해 인덱스에 0.5를 더한 값을 기준으로 계산합니다.
    """
    return (ix + 0.5) * RES, (iy + 0.5) * RES


def is_collision_point(x, y):
    """
    단일 점 (x,y)이 장애물 내부인지 확인
    
    주어진 미터 좌표를 그리드 인덱스로 변환한 뒤, 점유 그리드 지도(occupancy_grid)의
    해당 셀 값이 100(장애물 존재) 이상인지 확인합니다.
    """
    ix, iy = m_to_idx(x, y)
    return occupancy_grid[iy, ix] >= 100


def is_collision_line(x1, y1, x2, y2, n_checks=20):
    """
    (x1,y1) → (x2,y2) 직선 경로 상에 장애물이 있는지 확인.
    n_checks개의 점을 샘플링하여 충돌 검사.
    
    두 노드 사이의 직선 상에서 n_checks만큼의 지점을 선형 보간(np.linspace)하여
    각 샘플링 포인트가 장애물과 충돌하는지 개별적으로 검사합니다.
    """
    for t in np.linspace(0, 1, n_checks):
        x = x1 + t * (x2 - x1)
        y = y1 + t * (y2 - y1)
        if is_collision_point(x, y):
            return True
    return False


def calc_distance(n1, n2):
    """두 노드 사이의 유클리드 거리"""
    return np.hypot(n1.x - n2.x, n1.y - n2.y)


def calc_distance_pos(x1, y1, x2, y2):
    """두 좌표 사이의 유클리드 거리"""
    return np.hypot(x1 - x2, y1 - y2)


print("✅ Step 2 완료 — RRT* 파라미터 및 충돌 체크 함수 설정")
print(f"   MAX_ITER: {MAX_ITER}")
print(f"   STEP_SIZE (Δq): {STEP_SIZE} m")
print(f"   GOAL_THRESHOLD: {GOAL_THRESHOLD} m")
print(f"   GOAL_SAMPLE_RATE: {GOAL_SAMPLE_RATE}%")
print(f"   REWIRE_RADIUS: {REWIRE_RADIUS} m  ← RRT* 전용")

## Step 3 — RRT* 단계별 동작 시각화 (핵심!)

RRT*의 6단계를 하나씩 보여줍니다:
1. **샘플링** — 무작위 $q_{rand}$ 생성
2. **최근접 노드** — 트리에서 $q_{rand}$에 가장 가까운 $q_{near}$ 탐색
3. **확장** — $q_{near}$ → $q_{rand}$ 방향으로 $\Delta q$ 만큼 $q_{new}$ 생성
4. **충돌 체크** — 직선 경로에 장애물이 없는지 확인
5. **최적 부모 선택 (Choose Parent)** — $q_{new}$ 주변 반경 내 노드들 중 cost가 최소인 노드를 부모로 선택
6. **재배선 (Rewire)** — $q_{new}$를 통해 기존 주변 노드들의 cost가 줄어들면 부모를 변경

아래 코드는 **처음 5회 확장**을 하나씩 그려 RRT* 동작 원리를 직관적으로 보여줍니다.  
특히 **Choose Parent**와 **Rewire** 단계에서 RRT와의 차이를 강조합니다.

In [ ]:

def draw_rrt_star_step(ax, nodes, edges, title, start_m, goal_m,
                       q_rand=None, q_near=None, q_new=None,
                       near_nodes=None, best_parent=None, rewired=None,
                       collision=False, accepted=False):
    """
    RRT* 단계별 시각화.
    - nodes: RRTStarNode 리스트
    - edges: (parent_idx, child_idx) 리스트
    - q_rand, q_near, q_new: 현재 단계에서의 3단계 좌표
    - near_nodes: q_new 주변 반경 내의 노드 인덱스 리스트
    - best_parent: 최종 선택된 최적의 부모 노드 인덱스
    - rewired: (node_idx, old_parent, new_parent) 리스트
    - collision: 충돌 발생 여부
    - accepted: 트리에 추가 성공 여부
    """
    # 배경: 맵
    ax.imshow(occupancy_grid, origin='lower', cmap='gray_r',
              extent=[0, MAP_W, 0, MAP_H], vmin=0, vmax=100, alpha=0.5)

    # 기존 트리 엣지
    for p_idx, c_idx in edges:
        pn = nodes[p_idx]
        cn = nodes[c_idx]
        ax.plot([pn.x, cn.x], [pn.y, cn.y], 'c-', linewidth=1.0, alpha=0.6)

    # 기존 트리 노드
    if nodes:
        xs = [n.x for n in nodes]
        ys = [n.y for n in nodes]
        ax.scatter(xs, ys, c='cyan', s=20, zorder=3, alpha=0.7)

    # 시작점 강조
    ax.scatter(*start_m, c='lime', s=250, marker='o',
               edgecolors='black', zorder=6, label='Start ($q_{start}$)')
    ax.scatter(*goal_m, c='red', s=250, marker='X',
               edgecolors='black', zorder=6, label='Goal ($q_{goal}$)')

    # --- 현재 단계 시각화 ---
    if q_rand is not None:
        ax.scatter(*q_rand, c='magenta', s=120, marker='*',
                   zorder=7, label='$q_{rand}$', edgecolors='black')
    if q_near is not None:
        ax.scatter(*q_near, c='orange', s=120, marker='s',
                   zorder=7, label='$q_{near}$', edgecolors='black')
    
    # 주변 후보 노드들 (Rewire Radius 내)
    if near_nodes is not None and accepted:
        for n_idx in near_nodes:
            if n_idx < len(nodes):
                nn = nodes[n_idx]
                ax.scatter(nn.x, nn.y, c='yellow', s=80, marker='h',
                           zorder=4, alpha=0.5)
        # 반경 원
        if q_new is not None:
            circle = Circle(q_new, REWIRE_RADIUS, fill=False,
                            linestyle='--', color='yellow', linewidth=1.5, alpha=0.6)
            ax.add_patch(circle)

    if q_new is not None and q_near is not None:
        # 확장 방향 선 (q_near → q_rand)
        ax.annotate('', xy=q_rand, xytext=q_near,
                    arrowprops=dict(arrowstyle='->', color='gray',
                                    lw=1.5, ls='--'))
        # 실제 확장 선 (q_near → q_new) — 초록/빨강
        color = 'green' if accepted else 'red'
        label = '$q_{new}$ (Accepted)' if accepted else '$q_{new}$ (Collision)' if collision else '$q_{new}$'
        ax.plot([q_near[0], q_new[0]], [q_near[1], q_new[1]],
                color=color, linewidth=3.0, zorder=5)
        ax.scatter(*q_new, c=color, s=120, marker='D',
                   zorder=7, label=label, edgecolors='black')

    # 최종 선택된 부모 연결 (강조)
    if best_parent is not None and accepted and q_new is not None:
        pn = nodes[best_parent]
        ax.plot([pn.x, q_new[0]], [pn.y, q_new[1]],
                'g-', linewidth=3.5, zorder=6, label='Best Parent Path')
        ax.scatter(pn.x, pn.y, c='green', s=120, marker='P',
                   zorder=7, edgecolors='black', label='Best Parent')

    # Rewire된 엣지들 (파란색으로 강조)
    if rewired is not None and accepted:
        for node_idx, old_p, new_p in rewired:
            if node_idx < len(nodes):
                n = nodes[node_idx]
                # 새 부모와의 연결
                np_node = nodes[new_p]
                ax.plot([np_node.x, n.x], [np_node.y, n.y],
                        'b-', linewidth=2.5, zorder=5, alpha=0.8, label='Rewired' if node_idx == rewired[0][0] else '')
                ax.scatter(n.x, n.y, c='blue', s=100, marker='v',
                           zorder=6, edgecolors='black')

    ax.set_title(title, fontsize=10)
    ax.set_xlim(0, MAP_W); ax.set_ylim(0, MAP_H)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.2)
    ax.legend(loc='upper right', fontsize=7)


def run_rrt_star_step_by_step(max_steps=5):
    """
    RRT*를 처음 max_steps회 확장하면서 매 단계를 시각화.
    반환: nodes, edges (이후 재사용 가능)
    """
    nodes = [RRTStarNode(START_M[0], START_M[1], None)]
    edges = []

    fig, axes = plt.subplots(1, max_steps, figsize=(4*max_steps, 4))
    if max_steps == 1:
        axes = [axes]

    step_drawn = 0
    # 난수 생성기(RNG) 객체 생성 및 시드(Seed)를 42로 고정하여 매 실행마다 동일한 난수 경로가 생성되도록 재현성 보장
    rng = np.random.default_rng(42)

    for i in range(MAX_ITER):
        if step_drawn >= max_steps:
            break

        # -------------------------------------------------
        # 1. 샘플링 (q_rand)
        # -------------------------------------------------
        if rng.random() * 100 < GOAL_SAMPLE_RATE:
            q_rand = np.array([GOAL_M[0], GOAL_M[1]])
        else:
            q_rand = np.array([
                rng.uniform(0, MAP_W),
                rng.uniform(0, MAP_H)
            ])

        # -------------------------------------------------
        # 2. 가장 가까운 노드 탐색 (q_near)
        # -------------------------------------------------
        distances = [np.hypot(n.x - q_rand[0], n.y - q_rand[1]) for n in nodes]
        near_idx = int(np.argmin(distances))
        q_near = nodes[near_idx]

        # -------------------------------------------------
        # 3. 확장 (q_new)
        # -------------------------------------------------
        direction = q_rand - np.array([q_near.x, q_near.y])
        dist = np.hypot(direction[0], direction[1])
        if dist < 1e-6:
            continue
        direction = direction / dist
        step = min(STEP_SIZE, dist)
        q_new_pos = np.array([q_near.x, q_near.y]) + direction * step

        # 맵 경계 클리핑
        q_new_pos[0] = np.clip(q_new_pos[0], 0, MAP_W)
        q_new_pos[1] = np.clip(q_new_pos[1], 0, MAP_H)

        # -------------------------------------------------
        # 4. 충돌 체크
        # -------------------------------------------------
        collision = is_collision_line(q_near.x, q_near.y, q_new_pos[0], q_new_pos[1])

        if collision:
            title = (f"Step {step_drawn+1}: Collision!\n"
                     f"iter={i+1} | q_rand=({q_rand[0]:.2f},{q_rand[1]:.2f})")
            draw_rrt_star_step(
                axes[step_drawn], nodes, edges, title,
                START_M, GOAL_M,
                q_rand=tuple(q_rand), q_near=(q_near.x, q_near.y),
                q_new=tuple(q_new_pos), collision=True, accepted=False)
            step_drawn += 1
            continue

        # -------------------------------------------------
        # 5. Choose Parent (RRT* 핵심 1)
        #    q_new 주변 반경 내의 노드들 중 cost가 최소인 노드를 부모로 선택
        # -------------------------------------------------
        near_indices = []
        for idx, n in enumerate(nodes):
            if calc_distance_pos(n.x, n.y, q_new_pos[0], q_new_pos[1]) <= REWIRE_RADIUS:
                near_indices.append(idx)

        # 기본: q_near를 부모로
        min_cost = q_near.cost + calc_distance(q_near, RRTStarNode(q_new_pos[0], q_new_pos[1]))
        best_parent_idx = near_idx

        # 주변 노드들 중에서 더 좋은 부모가 있는지 탐색
        for idx in near_indices:
            n = nodes[idx]
            d = calc_distance_pos(n.x, n.y, q_new_pos[0], q_new_pos[1])
            if not is_collision_line(n.x, n.y, q_new_pos[0], q_new_pos[1]):
                new_cost = n.cost + d
                if new_cost < min_cost:
                    min_cost = new_cost
                    best_parent_idx = idx

        # -------------------------------------------------
        # 6. Rewire (RRT* 핵심 2)
        #    q_new를 통해 기존 노드들의 cost가 줄어들면 부모 변경
        # -------------------------------------------------
        rewired = []
        new_node = RRTStarNode(q_new_pos[0], q_new_pos[1], best_parent_idx)
        new_node.cost = nodes[best_parent_idx].cost + calc_distance(nodes[best_parent_idx], new_node)
        new_idx = len(nodes)

        for idx in near_indices:
            if idx == best_parent_idx:
                continue
            n = nodes[idx]
            d = calc_distance(new_node, n)
            # q_new를 통해 가는 것이 더 짧으면 재배선
            if new_node.cost + d < n.cost:
                if not is_collision_line(new_node.x, new_node.y, n.x, n.y):
                    old_parent = n.parent_idx
                    n.parent_idx = new_idx
                    n.cost = new_node.cost + d
                    # 엣지 업데이트: 기존 (old_parent, idx) 제거, (new_idx, idx) 추가
                    edges = [(p, c) for (p, c) in edges if c != idx]
                    edges.append((new_idx, idx))
                    rewired.append((idx, old_parent, new_idx))

        # 트리에 추가
        nodes.append(new_node)
        edges.append((best_parent_idx, new_idx))

        title = (f"Step {step_drawn+1}: Extended + Rewire\n"
                 f"iter={i+1} | q_new=({q_new_pos[0]:.2f},{q_new_pos[1]:.2f})")

        draw_rrt_star_step(
            axes[step_drawn], nodes[:-1], edges[:-1], title,
            START_M, GOAL_M,
            q_rand=tuple(q_rand), q_near=(q_near.x, q_near.y),
            q_new=tuple(q_new_pos), near_nodes=near_indices,
            best_parent=best_parent_idx, rewired=rewired,
            collision=False, accepted=True)

        step_drawn += 1

    plt.suptitle("Step 3 — RRT* Step-by-Step (Sampling → Near → Extend → ChooseParent → Rewire)",
                 fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()
    return nodes, edges


nodes_demo, edges_demo = run_rrt_star_step_by_step(max_steps=5)
print(f"\n✅ Step 3 완료 — RRT* 5단계 확장 시각화")
print(f"   현재 트리 노드 수: {len(nodes_demo)}")
print(f"   현재 트리 엣지 수: {len(edges_demo)}")

## Step 4 — RRT* 전체 실행 & 트리 성장 과정 시각화

이제 RRT*를 완전히 실행하여 목적지까지 도달할 때까지 트리를 성장시킵니다.  
중간 스냅샷(25%, 50%, 75%, 100%)을 통해 **Rewiring으로 인한 점진적 최적화**를 관찰합니다.

> RRT와 달리 RRT*는 반복이 진행될수록 **기존 엣지들이 더 짧은 연결로 교체되어** 트리가 점점 최적에 가까워집니다.

In [ ]:

def run_rrt_star_full(save_snapshots=True):
    """
    RRT*를 전체 실행.
    save_snapshots=True 이면 중간 단계의 nodes/edges를 기록.
    """
    nodes = [RRTStarNode(START_M[0], START_M[1], None)]
    edges = []
    rng = np.random.default_rng(42)

    goal_reached = False
    goal_node_idx = None

    # 트리의 중간 상태(Deep Copy)를 정기적으로 저장해 둘 임시 리스트
    temp_snapshots = []

    for i in range(MAX_ITER):
        # 1. 샘플링
        if rng.random() * 100 < GOAL_SAMPLE_RATE:
            q_rand = np.array([GOAL_M[0], GOAL_M[1]])
        else:
            q_rand = np.array([rng.uniform(0, MAP_W), rng.uniform(0, MAP_H)])

        # 2. 최근접 노드
        distances = [np.hypot(n.x - q_rand[0], n.y - q_rand[1]) for n in nodes]
        near_idx = int(np.argmin(distances))
        q_near = nodes[near_idx]

        # 3. 확장
        direction = q_rand - np.array([q_near.x, q_near.y])
        dist = np.hypot(direction[0], direction[1])
        if dist < 1e-6:
            continue
        direction = direction / dist
        step = min(STEP_SIZE, dist)
        q_new_pos = np.array([q_near.x, q_near.y]) + direction * step
        q_new_pos[0] = np.clip(q_new_pos[0], 0, MAP_W)
        q_new_pos[1] = np.clip(q_new_pos[1], 0, MAP_H)

        # 4. 충돌 체크
        if is_collision_line(q_near.x, q_near.y, q_new_pos[0], q_new_pos[1]):
            continue

        # 5. Choose Parent
        near_indices = []
        for idx, n in enumerate(nodes):
            if calc_distance_pos(n.x, n.y, q_new_pos[0], q_new_pos[1]) <= REWIRE_RADIUS:
                near_indices.append(idx)

        min_cost = q_near.cost + calc_distance(q_near, RRTStarNode(q_new_pos[0], q_new_pos[1]))
        best_parent_idx = near_idx

        for idx in near_indices:
            n = nodes[idx]
            d = calc_distance_pos(n.x, n.y, q_new_pos[0], q_new_pos[1])
            if not is_collision_line(n.x, n.y, q_new_pos[0], q_new_pos[1]):
                new_cost = n.cost + d
                if new_cost < min_cost:
                    min_cost = new_cost
                    best_parent_idx = idx

        # 6. Rewire
        new_node = RRTStarNode(q_new_pos[0], q_new_pos[1], best_parent_idx)
        new_node.cost = nodes[best_parent_idx].cost + calc_distance(nodes[best_parent_idx], new_node)
        new_idx = len(nodes)

        for idx in near_indices:
            if idx == best_parent_idx:
                continue
            n = nodes[idx]
            d = calc_distance(new_node, n)
            if new_node.cost + d < n.cost:
                if not is_collision_line(new_node.x, new_node.y, n.x, n.y):
                    n.parent_idx = new_idx
                    n.cost = new_node.cost + d
                    edges = [(p, c) for (p, c) in edges if c != idx]
                    edges.append((new_idx, idx))

        nodes.append(new_node)
        edges.append((best_parent_idx, new_idx))

        # 성공적인 확장 시점마다 정기적으로 트리 상태를 복제하여 백업 (20스텝 간격)
        if save_snapshots and (i % 20 == 0):
            snap_nodes = []
            for n in nodes:
                cn = RRTStarNode(n.x, n.y, n.parent_idx)
                cn.cost = n.cost
                snap_nodes.append(cn)
            snap_edges = [e for e in edges]
            temp_snapshots.append((i, snap_nodes, snap_edges))

        # 7. 목적지 도달 체크
        d_to_goal = np.hypot(q_new_pos[0] - GOAL_M[0], q_new_pos[1] - GOAL_M[1])
        if d_to_goal <= GOAL_THRESHOLD:
            goal_reached = True
            goal_node_idx = new_idx
            if save_snapshots:
                # 목적지 최종 도달 시점의 완전한 트리 복제 저장
                snap_nodes = []
                for n in nodes:
                    cn = RRTStarNode(n.x, n.y, n.parent_idx)
                    cn.cost = n.cost
                    snap_nodes.append(cn)
                snap_edges = [e for e in edges]
                temp_snapshots.append((i, snap_nodes, snap_edges))
            break

    # 도달하지 못하고 끝까지 실행된 경우 마지막 상태 저장
    if save_snapshots and not goal_reached:
        snap_nodes = []
        for n in nodes:
            cn = RRTStarNode(n.x, n.y, n.parent_idx)
            cn.cost = n.cost
            snap_nodes.append(cn)
        snap_edges = [e for e in edges]
        temp_snapshots.append((MAX_ITER, snap_nodes, snap_edges))

    # 백업된 이력 중 25%, 50%, 75%, 100% 시점의 균등한 4개 스냅샷 추출
    snapshots = []
    if save_snapshots and len(temp_snapshots) > 0:
        T = len(temp_snapshots)
        idx_25 = max(0, T // 4 - 1)
        idx_50 = max(0, T // 2 - 1)
        idx_75 = max(0, T * 3 // 4 - 1)
        idx_100 = T - 1

        selected_indices = [idx_25, idx_50, idx_75, idx_100]
        for idx in selected_indices:
            snapshots.append(temp_snapshots[idx])

    return nodes, edges, snapshots, goal_reached, goal_node_idx


print("🔄 RRT* 전체 실행 중...")
t0 = time.perf_counter()
nodes_full, edges_full, snapshots, ok_rrt_star, goal_idx = run_rrt_star_full()
elapsed_rrt_star = time.perf_counter() - t0

print(f"✅ Step 4 완료 — RRT* 전체 실행")
print(f"   목적지 도달: {'성공' if ok_rrt_star else '실패'}")
print(f"   실행 시간: {elapsed_rrt_star*1000:.2f} ms")
print(f"   총 노드 수: {len(nodes_full)}")
print(f"   총 엣지 수: {len(edges_full)}")
print(f"   Goal Bias 확률: {GOAL_SAMPLE_RATE}%")
print(f"   Rewire Radius: {REWIRE_RADIUS} m")

### Step 4-1 — RRT* 트리 성장 스냅샷

아래 4개의 스냅샷을 통해 RRT*가 어떻게 **Rewiring을 통해 점진적으로 최적화**되는지 확인합니다.  
초기에는 RRT와 유사하게 보이지만, 반복이 진행될수록 **더 짧은 부모 연결**로 교체되어 트리가 최적에 가까워집니다.

In [ ]:

def draw_rrt_star_snapshot(ax, nodes, edges, title, start_m, goal_m,
                           path_nodes=None):
    """RRT* 스냅샷 그리기"""
    # 배경
    ax.imshow(occupancy_grid, origin='lower', cmap='gray_r',
              extent=[0, MAP_W, 0, MAP_H], vmin=0, vmax=100, alpha=0.4)

    # 트리 엣지
    for p_idx, c_idx in edges:
        pn = nodes[p_idx]
        cn = nodes[c_idx]
        ax.plot([pn.x, cn.x], [pn.y, cn.y], 'c-', linewidth=0.8, alpha=0.5)

    # 트리 노드
    xs = [n.x for n in nodes]
    ys = [n.y for n in nodes]
    ax.scatter(xs, ys, c='cyan', s=12, zorder=3, alpha=0.6)

    # 경로 강조
    if path_nodes:
        px = [n.x for n in path_nodes]
        py = [n.y for n in path_nodes]
        ax.plot(px, py, 'g-', linewidth=3.0, marker='o', markersize=3,
                label='Path', zorder=5)

    ax.scatter(*start_m, c='lime', s=200, marker='o',
               edgecolors='black', zorder=6, label='Start')
    ax.scatter(*goal_m, c='red', s=200, marker='X',
               edgecolors='black', zorder=6, label='Goal')
    ax.set_title(title, fontsize=10)
    ax.set_xlim(0, MAP_W); ax.set_ylim(0, MAP_H)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.2)
    ax.legend(loc='upper right', fontsize=7)


def reconstruct_path(nodes, goal_idx):
    """목적지 노드부터 시작 노드까지 역추적"""
    path = []
    idx = goal_idx
    while idx is not None:
        path.append(nodes[idx])
        idx = nodes[idx].parent_idx
    return path[::-1]   # start → goal 순서로 반전


# 스냅샷 시각화
path_nodes = reconstruct_path(nodes_full, goal_idx) if ok_rrt_star else None

fig, axes = plt.subplots(2, 2, figsize=(11, 10))
axes = axes.flatten()

labels = ["25% iter", "50% iter", "75% iter", "100% (Goal Reached)"]
for i, (iter_n, snap_nodes, snap_edges) in enumerate(snapshots):
    draw_rrt_star_snapshot(axes[i], snap_nodes, snap_edges,
                           f"{labels[i]} — nodes={len(snap_nodes)}",
                           START_M, GOAL_M,
                           path_nodes=path_nodes if i == len(snapshots)-1 else None)

plt.suptitle("Step 4-1 — RRT* Tree Growth Snapshots (Choose Parent + Rewire Optimization)",
             fontsize=13, y=1.00)
plt.tight_layout()
plt.show()

if path_nodes:
    path_len = sum(
        np.hypot(path_nodes[i].x - path_nodes[i-1].x,
                 path_nodes[i].y - path_nodes[i-1].y)
        for i in range(1, len(path_nodes)))
    print(f"   최종 경로 길이: {path_len:.2f} m")
    print(f"   경로 노드 수: {len(path_nodes)}")
else:
    print("   경로를 찾지 못했습니다.")

## Step 5 — 최종 경로 & 전체 트리 시각화

완성된 트리 위에서 최종 경로를 녹색으로 강조 표시합니다.  
RRT*는 Rewiring을 통해 **점진적으로 최적 경로에 수렴**하므로, RRT보다 더 짧고 부드러운 경로를 생성합니다.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

# 배경
ax.imshow(occupancy_grid, origin='lower', cmap='gray_r',
          extent=[0, MAP_W, 0, MAP_H], vmin=0, vmax=100, alpha=0.3)

# 전체 트리
for p_idx, c_idx in edges_full:
    pn = nodes_full[p_idx]
    cn = nodes_full[c_idx]
    ax.plot([pn.x, cn.x], [pn.y, cn.y], 'c-', linewidth=0.6, alpha=0.4)

# 트리 노드
xs = [n.x for n in nodes_full]
ys = [n.y for n in nodes_full]
ax.scatter(xs, ys, c='cyan', s=8, zorder=3, alpha=0.5, label='Tree Nodes')

# 최종 경로
if path_nodes:
    px = [n.x for n in path_nodes]
    py = [n.y for n in path_nodes]
    ax.plot(px, py, 'g-', linewidth=4.0, marker='o', markersize=5,
            label='RRT* Path', zorder=5)

ax.scatter(*START_M, c='lime', s=300, marker='o',
           edgecolors='black', zorder=6, label='Start')
ax.scatter(*GOAL_M, c='red', s=300, marker='X',
           edgecolors='black', zorder=6, label='Goal')

ax.set_title("Step 5 — Final RRT* Tree & Optimized Path", fontsize=12)
ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
ax.set_xlim(0, MAP_W); ax.set_ylim(0, MAP_H)
ax.set_aspect('equal'); ax.grid(True, alpha=0.2)
ax.legend(loc='upper right', fontsize=9)

plt.tight_layout()
plt.show()

print("✅ Step 5 완료 — 최종 경로 시각화")